In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import geopandas as gpd
import matplotlib.patheffects as pe
import utils

In [ ]:
df = pd.read_csv('../data/Raw/Louisville_Metro_KY_-_Library_Collection_Inventory_.csv')

In [ ]:
df.head()

### Cleaning the Data  
The dataset contained several issues that must be addressed before analysis:  
- The 'BibNum', 'Author', 'ISBN', 'PublicationDate', and 'ReportDate' (if it is one date throughout the column) columns needed to be dropped as they were not necessary to answer questions I have about the data.
- Renamed leftover columns to match syntax used in other notebooks and sqlite.

In [ ]:
df.info()

In [ ]:
df['ReportDate'].unique()

In [ ]:
df.drop(['ReportDate'], axis=1, inplace=True)

In [ ]:
df.drop(['BibNum','Author', 'ISBN', 'PublicationYear'], axis=1, inplace=True)

In [ ]:
df = df.rename(columns={'ObjectId': 'item_id', 'Title':'title', 'ItemType':'item_type', 'ItemCollection': 'item_collection', 'ItemLocation':'item_location', 'ItemPrice':'item_price'})

In [ ]:
df.info()

### EDA
Missing data was handled as shown below:
- The row with 'Title' == NaN was dropped as it contains very little data.
- Due to a very low percentage of 'ItemCollection' == NaN and the wide variety of options regarding 'ItemCollection', it was left as is.
- For instances where 'ItemPrice' is equal to 0, it was left as is since there is no way to verify from the data if the data is missing or is equal to 0 as not all items in the library collection have an item price (donations).

Any visualizations using columns with missing data will be noted in the visualization markdown.

Regarding duplicated rows, rows cannot be validated as duplicated data or as simply duplicate items in the library inventory.

In [ ]:
df.isna()

df.isna().sum()

In [ ]:
df.loc[df['title'].isna()]

In [ ]:
df = df.drop(243965)

In [ ]:
df.isna().sum()

In [ ]:
df.loc[df['item_collection'].isna()]

In [ ]:
df.item_collection.unique()

In [ ]:
item_collection_na = df.item_collection.isna().sum()
total_rows = df.shape[0]
per_item_collection_na = (item_collection_na / total_rows) * 100
per_item_collection_na

In [ ]:
zero_item_price = df[df['item_price'] == 0]
zero_item_price

In [ ]:
df['item_price'] = df['item_price'].apply(lambda x: f'${x:,.2f}')

In [ ]:
df[df['item_price'] == 0.0]

In [ ]:
duplicated_mask = df.duplicated(keep=False)
duplicated_rows = df[duplicated_mask]
duplicated_rows

### The Geographic Distribution of LFPL Branches and OSS Households
- This section includes sql queries and visualizations to help with the analysis of the geographic coverage of the LFPL branches in regard to OSS Households.
- A Pin Map was a helpful visual when looking at how the library branches impact the geography of the Louisville area. A light grey background was chosen in order to highlight the smaller libraries with the lighter orange pins. The orange cmap selection allowed the pins to be more visible. 


In [ ]:
connection = sqlite3.connect("../database/lfpl_oss_household_demographics.db")
cursor = connection.cursor()

In [ ]:
#Percentage of Zipcodes with a Library
query1 = pd.read_sql('''
SELECT COUNT(DISTINCT l.zipcode) AS zipcodes_with_library,
    COUNT(DISTINCT lz.zipcode) AS total_louisville_zipcodes,
    COUNT(DISTINCT l.zipcode) * 100 / COUNT(DISTINCT lz.zipcode) AS library_zipcodes_pct
FROM libraries l
RIGHT JOIN louisville_zipcodes lz ON l.zipcode = lz.zipcode;
''', connection)
query1

In [ ]:
#Percentage of Zipcodes with OSS Households
query2 = pd.read_sql('''
SELECT COUNT(DISTINCT o.zipcode) AS oss_household_zipcodes,
    COUNT(DISTINCT lz.zipcode) AS total_louisville_zipcodes,
    COUNT(DISTINCT o.zipcode) * 100 / COUNT(DISTINCT lz.zipcode) AS oss_zipcode_pct
FROM oss_households o
RIGHT JOIN louisville_zipcodes lz ON o.zipcode = lz.zipcode;
''', connection)
query2

In [ ]:
#Percentage of OSS Zipcodes with a Library
query3 = pd.read_sql('''
SELECT COUNT(DISTINCT l.zipcode) AS zipcodes_with_library,
    COUNT(DISTINCT o.zipcode) AS oss_household_zipcodes,
    COUNT(DISTINCT l.zipcode) * 100 / COUNT(DISTINCT o.zipcode) AS library_oss_zipcode_pct
FROM oss_households o
LEFT JOIN libraries l ON o.zipcode = l.zipcode;
''', connection)
query3

In [ ]:
lfpl_loc = gpd.read_file("../data/Raw/Louisville_KY_Free_Public_Libraries/Louisville_KY_Free_Public_Libraries.shp")
zipcodes = gpd.read_file("../data/Raw/Jefferson_County_KY_ZIP_Codes/Jefferson_County_KY_ZIP_Codes.shp")

In [ ]:
query4 = pd.read_sql('''
SELECT zipcode, COUNT(*) AS oss_households
FROM oss_households
GROUP BY zipcode;
''', connection)
query4

In [ ]:
oss_households_map = zipcodes.merge(query4, left_on='ZIPCODE', right_on='zipcode', how='left')

In [ ]:
ax = oss_households_map.plot(
    column='oss_households',
    cmap='Oranges',
    figsize=(10, 10),
    edgecolor='black',
    legend=True,
    missing_kwds={"color": "lightgrey"}
)

ax.set_title('OSS Households by ZIP Code', fontsize=12)
ax.axis('off')

plt.savefig('../plots/OSSHouseholdsMap.png')
plt.show()

Zipcodes that contain OSS Households make up 92% of Jefferson County; however, the majority of households reside in the Western portion of the county as noted by the darker orange zipcodes shown above.

In [ ]:
#OSS Household Count per Library Branch
query5 = pd.read_sql('''
SELECT 
    o.zipcode,
    l.library_name,
    COUNT(o.household_id) AS household_count
FROM libraries l
JOIN oss_households o 
    ON l.zipcode = o.zipcode
GROUP BY o.zipcode, l.library_name
ORDER BY household_count DESC;
''', connection)
query5

In [ ]:
utils.name_fix(query5, 'library_name', lfpl_loc, 'LFPL_NAME')

In [ ]:
households_map = lfpl_loc.merge(query5, left_on='LFPL_NAME', right_on='library_name', how='left')

In [ ]:
utils.lfpl_map(zipcodes, households_map, 'household_count', 'How Many OSS Househoulds Are Being Served By Each LFPL Branch?', pe, plt, '../plots/OSSHouseholdsPerBranchMap.png')


- Important note: Household counts are reported at the zipcode level. Libraries within the same zipcode—such as Parkland and Shawnee (40211) and Main and Western (40203)—display identical values.
- There is a correlation between zipcodes with the highest amount of OSS Households and zipcodes containing a library branch.

In [ ]:
#Average Annual Income of OSS Households per Library Branch
query6 = pd.read_sql('''
SELECT l.library_name, ROUND(AVG(o.annual_income), 2) as avg_oss_household_annual_income
FROM oss_households o
JOIN libraries l ON o.zipcode = l.zipcode
GROUP BY l.library_name
ORDER BY avg_oss_household_annual_income;
''', connection)
query6

In [ ]:
utils.name_fix(query6, 'library_name', lfpl_loc, 'LFPL_NAME')

In [ ]:
query6['avg_oss_household_annual_income'] = query6['avg_oss_household_annual_income'].astype('int64')

In [ ]:
annual_income_map = lfpl_loc.merge(query6, left_on='LFPL_NAME', right_on='library_name', how='left')

In [ ]:
utils.lfpl_map(zipcodes, annual_income_map, 'avg_oss_household_annual_income', 'What is the Average Annual Income for OSS Households Per LFPL Branch?', pe, plt, '../plots/OSSAnnualIncomePerBranchMap.png')


- Important note: Household counts are reported at the zipcode level. Libraries within the same zipcode—such as Parkland and Shawnee (40211) and Main and Western (40203)—display identical values.
- The area of the greatest need is the Northwest/Western portion of Jefferson County, as can be seen from the other visualizations as well.

In [ ]:
inventory = pd.read_csv('../data/Clean/clean_lfpl_inventory.csv')

In [ ]:
book_itemtype = inventory[inventory['item_type'] == 'Book']
book_count = book_itemtype.groupby('item_location').size().reset_index(name='books')

In [ ]:
book_count = book_count[book_count["item_location"] != "Bookmobile"]
book_count


The Bookmobile location was not included as it does not have a set geographic location.

In [ ]:
utils.name_fix(book_count, 'item_location', lfpl_loc, 'LFPL_NAME')

In [ ]:
book_count_map = lfpl_loc.merge(book_count, left_on="LFPL_NAME", right_on="item_location")

In [ ]:
utils.lfpl_map(zipcodes, book_count_map, 'books', "Which LFPL Branches Offer the Most Books?", pe, plt, '../plots/TotalBooksPinMap.png')

- Important Note: The data for the LFPL Inventory did not include information on the Parkland branch. 
- The Main branch has the largest collection items, which makes sense as it is the oldest branch and also holds ownership over many of the electronic items.
- The collections of individual branches are smaller in the Northwestern/Western portion of Jefferson County; however, there is a slightly higher concentration of branches located there (especially in the North). In addition, the smaller collections of branches such as Western, Shawnee, Parkland, and Portland may be aided by the largest collection at the Main branch.
- The larger branches, excepting Main, are located in areas less concentrated with other branches and can provide for larger geographic regions. These branches are not located in zipcodes with the highest concentrations of OSS Households, and their OSS Households hold higher annual incomes. 
- A new branch is opening in 2026 in the Fern Creek area between the South Central and Jeffersontown branches.

### The Distribution of the LPFL Collection
Bar charts were used in this section to show the distribution of the library collection as a whole and the book collection across all the library branches. A simple blue color was chosen for the bar charts used for the visualization folder. The goal was to keep the charts readable and focused on their objective.

In [ ]:
df['item_location'] = df['item_location'].replace('Remote Shelving - Shawnee', 'Shawnee')
df['item_location'] = df['item_location'].replace('Childrens Bookmobile', 'Bookmobile')
df['item_location'] = df['item_location'].replace('Adult Bookmobile', 'Bookmobile')
df['item_location'] = df['item_location'].replace(['Remote Shelving - Main', 'Childrens Main Library', 'Main Teen'], 'Main')

Some 'ItemLocation' entries needed to be combined to get a total for the branch (i.e. 'Remote Shelving-Shawnee' to 'Shawnee').

In [ ]:
df[df['item_location'] == 'Content Management'].value_counts()

In [ ]:
df['item_location'] = df['item_location'].replace('Content Management', 'Main')

The 'ItemLocation' called 'Content Management' was replaced with the standard location, 'Main,' as it only applied to 3 items.

In [ ]:
total_item_count = df['item_location'].value_counts()
total_item_count

plt.bar(total_item_count.index, total_item_count.values, color="#1A6D96")
plt.xticks(rotation=45, ha='right')

plt.xlabel('Library Branch')
plt.ylabel('Total Item Count')
plt.title('Total Amount of Items at Each Library Branch')

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/TotalItemAtEachBranchBar.png')
plt.show()

A different representation of the pin map above.

In [ ]:
items_per_branch = df.groupby('item_location')['item_type'].value_counts().reset_index().sort_values(by='count', ascending=True)
items_per_branch

In [ ]:
pivot_df = items_per_branch.pivot(
    index='item_location',
    columns='item_type',
    values='count'
).fillna(0)

In [ ]:
ax = pivot_df.plot(kind='bar', stacked=True, figsize=(10,6))

ax.set_title('Item Types By LFPL Branch')

ax.legend(
    title='Item Type',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

This scattered bar chart was not very helpful due to the large amount of item types, but it did help confirm that majority of the item types that are not books are located at the Main branch. 

In [ ]:
item_count_non_Main = (
    df[df['item_location'] != 'Main']['item_location']
    .value_counts())


plt.bar(item_count_non_Main.index, item_count_non_Main.values, color="#1A6D96")
plt.xticks(rotation=45, ha='right')

plt.xlabel('Library Branch')
plt.ylabel('Total Item Count')
plt.title('Total Amount of Items at Each LFPL Branch (Excluding Main)')

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/TotalItemAtEachBranchExceptMainBar.png')
plt.show()

This plot without the Main branch was included as the Main branch has a disproportionate amount of items compared to the other branches. 

In [ ]:
df.item_type.unique()

In [ ]:
book_itemtype = df[df['item_type'] == 'Book']
book_count = book_itemtype.groupby('item_location')['item_type'].size().sort_values(ascending=False)
book_count

plt.bar(book_count.index, book_count.values, color="#1A6D96")
plt.xticks(rotation=45, ha='right')

plt.xlabel('Library Branch')
plt.ylabel('Total Book Count')
plt.title('Total Amount of Books at Each Library Branch')

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/TotalBooksAtEachBranchBar.png')
plt.show()

This bar chart showing only book items displayed an exact trend as seen in the bar chart of all item types in which the Main branch has the vast majority of items and the branches went in the same order with book count.

### Comparisons within the LFPL Collection
- Heatmaps were used to show how the different item collections were disbursed across the different branches. 
- A simple pie chart showed a comparison of LFPL Ebook versus Physical Book holdings.

In [ ]:
df['item_collection'].unique()

In [ ]:
childrens_words = ['Children', 'Preschool', 'Storytime']
pattern = '|'.join(childrens_words)
childrens_df = df[
    df['item_collection'].str.contains(pattern, case=False, na=False)
]


In [ ]:
childrens_df_byloc = childrens_df.groupby(['item_location', 'item_collection']).size().reset_index(name='count')
childrens_df_byloc


In [ ]:
heatmap_data = childrens_df_byloc.pivot(
    index='item_location',
    columns='item_collection',
    values='count'
).fillna(0)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, cmap='Blues')

plt.title("Children's Collection Items by Library Branch")
plt.xlabel("Collection Type")
plt.ylabel("Library Branch")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../plots/ChildrensCollectionHeatMap.png')
plt.show()

- LFPL branches that hold the largest collections of Children's books include Main, Northeast, South Central, and Southwest.
- The genres with the highest amount of books are Non-Fiction, Fiction (including Paperback), and Picture Books.
- For areas with the highest concentration of OSS Households, the Main, Iroquois, and Shawnee branches may provide the best options for Children's books.

In [ ]:
teen_words = ['Teen', 'College']
pattern = '|'.join(teen_words)
teens_df = df[
    df['item_collection'].str.contains(pattern, case=False, na=False)]

In [ ]:
teens_df_byloc = teens_df.groupby(['item_location', 'item_collection']).size().reset_index(name='count')
teens_df_byloc

In [ ]:
heatmap_data = teens_df_byloc.pivot(
    index='item_location',
    columns='item_collection',
    values='count'
).fillna(0)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, cmap='Blues')

plt.title("Teen Collection Items by Library Branch")
plt.xlabel("Collection Type")
plt.ylabel("Library Branch")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../plots/TeenCollectionHeatMap.png')
plt.show()

- The Teen collection is more spread out more among the LFPL branches.
- LFPL branches that hold the largest collections of Teen books include South Central, Southwest, Northeast, Shawnee, and Main. Surprisingly, Main is the least of these branches to hold a larger collection.
- The genres with the highest amount of books are Non-Fiction and Fiction.
- For areas with the highest concentration of OSS Households, the Shawnee and Main branches may provide the best options for Teen books.

In [ ]:
adult_words = ['Adult', 'ELL', 'Mystery', 'Science', 'Western', 'International', 'Kentucky', 'Natural', 'Oversize','Holiday', 'Urban', 'Bestsellers', 'Large', 'Caldecott/Newberry', 'Government', 'Telereference', 'Big', 'Magazines']
pattern = '|'.join(adult_words)
adults_df = df[
    df['item_collection'].str.contains(pattern, case=False, na=False)]

In [ ]:
adults_df_byloc = adults_df.groupby(['item_location', 'item_collection']).size().reset_index(name='count')
adults_df_byloc

In [ ]:
heatmap_data = adults_df_byloc.pivot(
    index='item_location',
    columns='item_collection',
    values='count'
).fillna(0)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, cmap='Blues')

plt.title("Adult Collection Items by Library Branch")
plt.xlabel("Collection Type")
plt.ylabel("Library Branch")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../plots/AdultCollectionHeatMap.png')
plt.show()

- The Adult collection is much larger than the other collection which is expected.
- Since Main has the largest collection by far, it is difficult to discern much with a heat map.
- The use of the Main branch as an archive can be seen through its higher collection of Reference and Kentucky History books.

In [ ]:
adults_df_byloc_filtered = adults_df_byloc[adults_df_byloc['item_location'] != 'Main']


In [ ]:
heatmap_data = adults_df_byloc_filtered.pivot(
    index='item_location',
    columns='item_collection',
    values='count'
).fillna(0)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, cmap='Blues')

plt.title("Adult Collection Items by Library Branch (Except Main)")
plt.xlabel("Collection Type")
plt.ylabel("Library Branch")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../plots/AdultCollectionExceptMainHeatMap.png')
plt.show()

- Excluding Main, the LFPL branches that hold the largest collections of adult books include Southwest, South Central, Northeast, Bon Air, and St. Matthews. 
- The genres with the highest amount of books are Non-Fiction and Fiction.
- For areas with the highest concentration of OSS Households, the Main or Iroquois branches is best option for the largest collection of adult books.

In [ ]:
df.item_type.unique()

In [ ]:
ebooks = df[df['item_type'].isin(['Ebook'])]

In [ ]:
ebook_len = len(ebooks.value_counts())

In [ ]:
book_len = len(df[df['item_type'] == 'Book'].value_counts())

In [ ]:
plt.figure(figsize=(6,6))
plt.pie(
    [ebook_len, book_len],
    labels=None,
    startangle=90,
    colors=["#8fb8caff", "#22649aff"],
    autopct='%1.1f%%',
    textprops={'fontsize' : 12, 'weight': 'bold'})

plt.title('How Large of a Role Do Ebooks Have in Our Libraries?', fontsize=16)
plt.legend(
    labels = ['Ebooks', 'Physical books'], fontsize=12,
    loc='lower right'
)


plt.tight_layout()
plt.savefig('../plots/BooksvsEbooksPieChart.png')
plt.show()


This pie chart shows the amount of Ebooks relative to physical books included in the LFPL collection. A future goal for this project would be to include circulation data if possible and see how the Ebook collection has changed over the years. Growth in electronic resources such as Ebooks could be a benefit to those in OSS Households with limited income, transportation, or types of access to the library.

## Conclusions
- The LFPL branch distribution correlates well to zipcodes holding OSS Households. 
- While many of the LFPL branches serving the highest concentration of OSS Households have smaller collections, the Main branch with the largest collection supports the area as well. (Note: the Main branch is currently closed to renovations)
- The majority of the collection is housed at the Main branch, which makes sense as it is the oldest branch, houses the majority of electronic items, and serves a more archival role than the other branches.
- The Northeast, South Central, and Southwest branches have the largest collections (excluding Main). The Northeast and Southwest branches serve larger geographic regions with a smaller concentration of library branches. South Central has a bit more library branches in its area (with a new Fern Creek branch opening in 2026) but still serves a large geographic region.
- OSS Households in the Southwest portion of Jefferson County may have more resources at the South Central and Southwest branches, which hold large collections of adult, teen, and children's books.
- OSS Households in the Northwest portion of Jefferson County may have more resources at the Iroquois branch for adult books, the Shawnee branch for teen books, and either Iroquois or Shawnee for children's books. In addition, the Main branch holds large collections for adult, teen, and children's books.
- While requiring access to the internet, the Ebooks and other electronic options could be a helpful resource to OSS Household.

In [ ]:
df.to_csv('../data/Clean/clean_lfpl_inventory.csv', index=False)